# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaitanyamadire01-dotcom/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Archetype-to-Action Mapping and Reason Codes

To make model predictions actionable for editorial teams, pages are segmented into distinct archetypes with dedicated operational reason codes:

| Page Archetype | Condition / Signal Pattern | Action Label | Reason Code | Primary Intervention |
| :--- | :--- | :--- | :--- | :--- |
| **Striking Distance Stale** | Rank 4–10, Age > 180d, High Predicted Decay Probability | `REFRESH_SEARCH_INTENT` | `STRIKING_STALE_DECAY` | Update title, answer fresh sub-queries, refresh statistics |
| **Top Tier Decay Defense** | Rank 1–3, High Impressions, Showing Visibility Slip | `DEFEND_PAGE_ONE` | `PAGE_ONE_VULNERABILITY` | Audit competitor links, tighten intro, preserve schema |
| **Underperforming Giant** | Impressions > 2,000, CTR < 10th percentile for tier | `OPTIMIZE_SNIPPET` | `HIGH_IMP_LOW_CTR` | Rewrite meta title and description for higher search click capture |
| **Aging Anchor** | Age > 360d, Stable but high historical impression base | `SCHEDULED_AUDIT` | `LEGACY_ANNUAL_CHECK` | Check for broken links, out-of-date product specs, or dead assets |
| **Low-Yield Long-Tail** | Impressions < 100, Predicted Decay | `MONITOR_ONLY` | `LOW_OPPORTUNITY_HOLD` | Do not spend editorial bandwidth; backlog for bulk pruning |

In [1]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# 1. Load data
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv.")

df = pd.read_csv(path)
df = df[df['impressions_90d'] > 0].copy().reset_index(drop=True)
df['target_decay'] = (df['trend_direction'].str.lower() == 'down').astype(int)

# Honest features
df['is_striking'] = (df['position_tier'] == 'striking').astype(int)
df['is_page_1'] = (df['avg_position'] <= 10.0).astype(int)
df['log_impressions'] = np.log1p(df['impressions_90d'])
df['log_clicks'] = np.log1p(df['clicks_90d'])

features = ['avg_position', 'log_impressions', 'log_clicks', 'ctr', 'content_age_days', 'is_striking', 'is_page_1']
X = df[features]
y = df['target_decay']
groups = df['client_id']

# Grouped fit to generate out-of-fold calibration
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf = RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=25, random_state=42)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

# Generate decay probability across the full portfolio
df['decay_prob'] = rf.predict_proba(X)[:, 1]

# 2. Operational Action Assignment Function
def assign_action(row):
    prob = row['decay_prob']
    pos = row['avg_position']
    age = row['content_age_days']
    imp = row['impressions_90d']
    ctr = row['ctr']
    
    # Priority opportunity score combines probability of decay with volume at risk
    opp_score = prob * np.log1p(imp)
    
    if pos >= 4.0 and pos <= 10.0 and age > 180 and prob >= 0.50:
        return pd.Series([opp_score, 'REFRESH_SEARCH_INTENT', 'STRIKING_STALE_DECAY', 'High'], 
                         index=['action_priority_score', 'action_label', 'reason_code', 'effort_tier'])
    elif pos <= 3.0 and prob >= 0.45:
        return pd.Series([opp_score * 1.2, 'DEFEND_PAGE_ONE', 'PAGE_ONE_VULNERABILITY', 'Medium'], 
                         index=['action_priority_score', 'action_label', 'reason_code', 'effort_tier'])
    elif imp > 1500 and ctr < 0.12:
        return pd.Series([opp_score * 1.1, 'OPTIMIZE_SNIPPET', 'HIGH_IMP_LOW_CTR', 'Low'], 
                         index=['action_priority_score', 'action_label', 'reason_code', 'effort_tier'])
    elif age > 360 and prob >= 0.40:
        return pd.Series([opp_score * 0.8, 'SCHEDULED_AUDIT', 'LEGACY_ANNUAL_CHECK', 'Medium'], 
                         index=['action_priority_score', 'action_label', 'reason_code', 'effort_tier'])
    else:
        return pd.Series([opp_score * 0.3, 'MONITOR_ONLY', 'LOW_OPPORTUNITY_HOLD', 'None'], 
                         index=['action_priority_score', 'action_label', 'reason_code', 'effort_tier'])

action_df = df.apply(assign_action, axis=1)
df_playbook = pd.concat([df, action_df], axis=1)

# Sort ranked action queue
df_queue = df_playbook.sort_values(by='action_priority_score', ascending=False).reset_index(drop=True)

print("=== ACTION QUEUE DISTRIBUTION BY REASON CODE ===")
print(df_queue['reason_code'].value_counts())
print("\nTop 5 Priority Actions:")
url_col = 'url' if 'url' in df.columns else ('page_id' if 'page_id' in df.columns else df.columns[1])
preview_cols = ['client_id', url_col, 'action_label', 'reason_code', 'action_priority_score', 'decay_prob', 'impressions_90d']
preview_cols = [c for c in preview_cols if c in df_queue.columns]
print(df_queue[preview_cols].head(5).to_string(index=False))

=== ACTION QUEUE DISTRIBUTION BY REASON CODE ===
reason_code
LOW_OPPORTUNITY_HOLD      19476
STRIKING_STALE_DECAY       4333
HIGH_IMP_LOW_CTR           2975
LEGACY_ANNUAL_CHECK        2601
PAGE_ONE_VULNERABILITY      615
Name: count, dtype: int64

Top 5 Priority Actions:
        client_id         client_id     action_label            reason_code  action_priority_score  decay_prob  impressions_90d
client_6208ef0f77 client_6208ef0f77  DEFEND_PAGE_ONE PAGE_ONE_VULNERABILITY               9.983187    0.707406           128068
client_6208ef0f77 client_6208ef0f77 OPTIMIZE_SNIPPET       HIGH_IMP_LOW_CTR               9.112227    0.703211           130617
client_7f2253d7e2 client_7f2253d7e2 OPTIMIZE_SNIPPET       HIGH_IMP_LOW_CTR               9.015664    0.728548            76868
client_6208ef0f77 client_6208ef0f77 OPTIMIZE_SNIPPET       HIGH_IMP_LOW_CTR               9.011670    0.718295            89803
client_6208ef0f77 client_6208ef0f77 OPTIMIZE_SNIPPET       HIGH_IMP_LOW_CTR             

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended Use and Operational Boundaries

#### Intended Use (Decision-Support Only)
* **Target Audience**: Content strategists, SEO managers, and editorial sprint leads.
* **Operational Scope**: Weekly or monthly triage to prioritize editorial refreshing, meta-tag testing, and content consolidation sprints.
* **Function**: Directs scarce human editorial hours toward URLs with significant traffic volume at immediate risk of visibility loss, replacing arbitrary calendar-based audits.

#### Explicit Operating Limits
1. **Non-Causal Scoring**: The model estimates predictive correlation, not causal impact. Ranking a page high in the queue does not guarantee that rewrites will reverse traffic loss if search intent fundamentally shifted.
2. **Lagged Window**: Features rely on rolling 90-day search aggregates. Algorithmic penalties or catastrophic 404 outages occurring within the last 7–14 days are obscured and will not be caught immediately.
3. **Domain Transferability**: Scores calibrated across client groupings reflect generalized historical patterns. Client domains operating in distinct high-volatility niches (e.g., breaking news, event tickets) may show different decay tempos.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human Review Protocols and The "No-Go" Automation List

#### The Skeptic's Pre-Flight Review Checklist (Before Any Edit)
Prior to assigning an editorial ticket, a human reviewer must verify:
1. **SERP Layout Verification**: Did organic visibility drop because a Google AI Overview, Knowledge Panel, or sponsored ad block absorbed clicks above position 1?
2. **Cannibalization Check**: Has the client published a newer, competing article that is intentionally taking over this query cluster?
3. **Seasonality Audit**: Is the keyword tied to an annual holiday or event where demand naturally collapses after a seasonal peak?

#### The Strict No-Go List (NEVER Automate Without Editorial Sign-Off)
* **NO Programmatic Bulk Rewriting**: Never plug queued URLs directly into an LLM for autonomous content overwriting. AI-generated rehashes risk hallucinating factual errors, degrading brand voice, and triggering search quality penalties.
* **NO Automated URL Deletion or Redirection**: Deleting or 301-redirecting pages based solely on decay scores can destroy valuable legacy backlink equity or sever vital internal link topology.
* **NO Autonomous Title Changes on Regulated Pages**: Legal, terms-of-service, privacy, and compliance-sensitive pages must be permanently exempt from automated headline modifications.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Cost/Value Calculus and Monitoring/Retrain Protocols

#### Cost/Value Heuristic
* **Editorial Cost**: A full content rewrite demands 3–5 hours of editorial labor ($150–$300 equivalent cost). A snippet optimization requires 15 minutes ($20 equivalent cost).
* **Opportunity Threshold**: An editorial refresh is only assigned if the page represents $\ge 1,000$ ninety-day impressions where a 2-position recovery recovers sufficient click volume to offset labor expense. Low-impression URLs default strictly to `MONITOR_ONLY`.

#### Model Monitoring & Drift Triggers
The playbook must trigger a retraining and threshold review when:
1. **Precision@50 Drift**: Measured monthly precision on verified decay drops below **0.65** for two consecutive sprint cycles.
2. **SERP Structural Shifts**: Major search engine core algorithmic releases change baseline click-through distributions by more than 20%.
3. **Data Freshness / Contract Drift**: The warehouse schema updates, or the proportion of nulls / zero-impression rows diverges by $>15\%$ from baseline contract distributions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import matplotlib.pyplot as plt

# 1. Setup export paths
outputs_dir = Path("work/outputs")
figures_dir = Path("work/figures")
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# 2. Export Ranked Queue CSV (reproducible output for the paper)
csv_export_path = outputs_dir / "ranked_action_queue.csv"
export_columns = [
    'client_id', url_col, 'action_label', 'reason_code', 
    'action_priority_score', 'decay_prob', 'impressions_90d', 
    'avg_position', 'ctr', 'content_age_days', 'effort_tier'
]
export_columns = [c for c in export_columns if c in df_queue.columns]

df_queue[export_columns].to_csv(csv_export_path, index=False)
print(f"Exported ranked queue CSV: {csv_export_path} ({len(df_queue):,} rows)")

# 3. Export Summary Metrics JSON Receipt
metrics_receipt = {
    "total_pages_evaluated": int(len(df_queue)),
    "high_priority_actions_count": int((df_queue['effort_tier'] == 'High').sum()),
    "action_distribution": df_queue['action_label'].value_counts().to_dict(),
    "mean_decay_prob_high_priority": float(df_queue[df_queue['effort_tier'] == 'High']['decay_prob'].mean()),
    "min_impressions_for_refresh": 1000,
    "validation_split_strategy": "Grouped by client_id"
}

json_export_path = outputs_dir / "action_playbook_metrics.json"
with open(json_export_path, "w") as f:
    json.dump(metrics_receipt, f, indent=2)
print(f"Exported metrics JSON receipt: {json_export_path}")

# 4. Generate and Save Reusable Figure for the Research Paper
plt.figure(figsize=(9, 5))
action_counts = df_queue['action_label'].value_counts()
colors = ['#2b5c8f', '#d95f02', '#7570b3', '#1b9e77', '#e7298a']
action_counts.plot(kind='barh', color=colors[:len(action_counts)])
plt.title("Portfolio Distribution by Recommended Content Action", fontsize=12, fontweight='bold')
plt.xlabel("Number of Pages")
plt.ylabel("Action Label")
plt.tight_layout()

figure_path = figures_dir / "playbook_action_distribution.png"
plt.savefig(figure_path, dpi=300)
plt.close()
print(f"Exported publication figure: {figure_path}")

Exported ranked queue CSV: work\outputs\ranked_action_queue.csv (30,000 rows)
Exported metrics JSON receipt: work\outputs\action_playbook_metrics.json
Exported publication figure: work\figures\playbook_action_distribution.png


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Explains intended use and clear non-production operational limits.
- [x] Ranks actions with explicit reason codes and archetype mappings.
- [x] Establishes a human review checklist and strict no-go automation boundaries.
- [x] Details cost/value trade-offs and monitoring/retrain triggers.
- [x] Exports the ranked queue CSV and metrics JSON to work/outputs/.
- [x] Saves reusable publication figures to work/figures/.
- [x] Adheres to safe, measured decision-support language throughout.